# Mapeamento de Teses e Dissertações em Temas Estratégicos

SCC0634 - Aplicações de Inteligência Artificial

Prof. Ricardo M. Marcacini

Alunos:

* João Brasieiro Moreira da Silva - NºUSP: 13672957
* Lucas de Oliveira Ferreira - 13695042

#1. Baixando e importando bibliotecas

In [ ]:
!pip install gdown pandas pyarrow sentence-transformers "ollama>0.5.0" "openai==0.28" codecarbon -q

In [ ]:
import pandas as pd
import numpy as np
import gdown
import os
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import subprocess
import time
import openai
import json

# 2. Baixando e carregando dados

In [ ]:
#baixando os arquivos (instalar o pacote gdown)
!gdown 12H957uf6mK-1X_ztT9hgFS1slpN2j-Wh
!gdown 1-QXkqH8HzLcV2JCA4Nm9G5rQhorYKJVe

#path dos arquivos parquet
path_dict = "leandl_oesnpg_dicionario.parquet"
path_data = "leandl_oesnpg_dados.parquet"

# Leitura usando pandas
dicionario_df = pd.read_parquet(path_dict)
dados_df = pd.read_parquet(path_data)

print("Dimensões do dicionário:", dicionario_df.shape)
print("Dimensões dos dados:", dados_df.shape)

# Exibindo o dicionário para entendermos os campos
print("\nDicionário de Dados:")
display(dicionario_df)

Downloading...
From: https://drive.google.com/uc?id=12H957uf6mK-1X_ztT9hgFS1slpN2j-Wh
To: /content/leandl_oesnpg_dicionario.parquet
100% 4.24k/4.24k [00:00<00:00, 11.3MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1-QXkqH8HzLcV2JCA4Nm9G5rQhorYKJVe
From (redirected): https://drive.google.com/uc?id=1-QXkqH8HzLcV2JCA4Nm9G5rQhorYKJVe&confirm=t&uuid=09ad716c-8df0-4d43-818b-9248f878a936
To: /content/leandl_oesnpg_dados.parquet
100% 123M/123M [00:01<00:00, 117MB/s]
Dimensões do dicionário: (25, 2)
Dimensões dos dados: (42046, 25)

Dicionário de Dados:


,campo,descricao
0,hash_id,Identificador único (hash) para a produção aca...
1,tema_id,Identificador numérico único do tema estratégico.
2,tema,Nome do tema estratégico definido por uma Unid...
3,palavras_chave,Lista de palavras-chave associadas ao tema est...
4,uf_tema_info,Unidade da Federação (UF) responsável pela def...
5,uf_pesquisador,Unidade da Federação (UF) da instituição de ví...
6,nome_programa,Nome do programa de pós-graduação ao qual a pr...
7,sigla_entidade_ensino,Sigla oficial da instituição de ensino respons...
8,nome_producao,Título completo da tese ou dissertação.
9,nome_subtipo_producao,"Tipo de produção acadêmica, como tese (doutora..."


# 3. Geração de Embeddings

In [ ]:
# Carregar um modelo de embedding pré-treinado.
print("Carregando o modelo de sentence embeddings...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Modelo carregado.")

#Primeiro, criamos um DataFrame (`df_temas`) contendo apenas os temas únicos e suas palavras-chave para evitar processamento redundante. Em seguida, geramos um embedding para cada nome de tema.
def to_fset(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return frozenset(x)
    if pd.isna(x):
        return frozenset()
    return frozenset([x])

df_temas = (
    dados_df.assign(palavras_chave=dados_df["palavras_chave"].apply(to_fset))
            [["tema_id", "tema", "uf_tema_info", "palavras_chave"]]
            .drop_duplicates()
            .reset_index(drop=True)
)

print("\nGerando embeddings para os temas estratégicos...")

df_temas['tema_embedding'] = list(embedding_model.encode(df_temas.tema.to_list(), show_progress_bar=True))
print("Embeddings dos temas gerados com sucesso!")
display(df_temas.head())

# Agora, fazemos o mesmo para as produções acadêmicas. Para representar cada tese, concatenamos seu **título** e **resumo**.

# Criamos um DataFrame com as produções únicas e o texto a ser usado para embedding
df_producoes = dados_df[[
    'hash_id',
    'nome_producao',
    'descricao_resumo',
    'uf_pesquisador'
]].drop_duplicates(subset='hash_id').reset_index(drop=True)

# Concatenamos título e resumo para criar um texto consolidado
df_producoes['texto_completo'] = df_producoes['nome_producao'] + ". " + df_producoes['descricao_resumo']

print("\nGerando embeddings para as produções acadêmicas...")

# Para este exemplo, vamos processar apenas uma amostra para agilizar.
amostra_producoes_df = df_producoes
amostra_producoes_df['producao_embedding'] = list(embedding_model.encode(amostra_producoes_df.texto_completo.to_list(), show_progress_bar=True))
print("Embeddings das produções gerados com sucesso!")
display(amostra_producoes_df.head())

Carregando o modelo de sentence embeddings...
Modelo carregado.

Gerando embeddings para os temas estratégicos...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embeddings dos temas gerados com sucesso!


,tema_id,tema,uf_tema_info,palavras_chave,tema_embedding
0,1,Agronegócio e Tecnologias de Informação e Comu...,ACRE,"(inovação tecnológica, agroindústria, investim...","[0.35649076, 0.24630514, -0.046685662, -0.0893..."
1,3,Biodiversidade e Biotecnologia,ACRE,"(fomento, bioeconomia, inovação tecnológica, b...","[-0.11034125, 0.41178066, -0.006614129, -0.024..."
2,4,"Bioeconomia: cadeias produtivas prioritárias, ...",ACRE,"(bioeconomia, redução de emissões, desenvolvim...","[0.18388051, 0.12093152, 0.01948139, -0.011404..."
3,5,Desenvolvimento Regional da Tríplice Divisa Am...,ACRE,"(desenvolvimento sustentável, governança, Peru...","[0.44723502, -0.12672468, 0.24179825, -0.12053..."
4,6,Desenvolvimento Regional: integração Estado-Mu...,ACRE,"(ciência, tecnologia e inovação, políticas púb...","[0.41903856, -0.28475463, 0.5618403, -0.165873..."



Gerando embeddings para as produções acadêmicas...


Batches:   0%|          | 0/527 [00:00<?, ?it/s]

Embeddings das produções gerados com sucesso!


,hash_id,nome_producao,descricao_resumo,uf_pesquisador,texto_completo,producao_embedding
0,ce4025a58d1cff3d346e96af2e8f2d0185caeaddd78c1b...,AS TECNOLOGIAS DIGITAIS DA INFORMAÇÃO E COMUNI...,"A PESQUISA, DESCRITA NO PRESENTE TRABALHO, ENV...",ACRE,AS TECNOLOGIAS DIGITAIS DA INFORMAÇÃO E COMUNI...,"[0.013936554, 0.12597041, 0.0103796255, 0.0392..."
1,55982d77d62446fb9f76ae636c99c36d77d5e233ce4863...,O CURRÍCULO INTEGRADO DO INSTITUTO FEDERAL DO ...,ESTE ESTUDO INVESTIGOU O CURRÍCULO DO ENSINO M...,ACRE,O CURRÍCULO INTEGRADO DO INSTITUTO FEDERAL DO ...,"[0.016246615, 0.1790295, 0.076485895, -0.03354..."
2,1f7615b9be49f80d289ba7c99eb64a5f8dc387a23518a6...,TENDÊNCIA TEMPORAL E DISTRIBUIÇÃO ESPACIAL DAS...,"INTRODUÇÃO. AS ANÁLISES NOS BIOMAS: AMAZÔNIA, ...",ACRE,TENDÊNCIA TEMPORAL E DISTRIBUIÇÃO ESPACIAL DAS...,"[0.18606462, 0.0006506941, 0.113488406, 0.1689..."
3,3773fcb7084d294753146c9580750eca9b2348b78cc4aa...,MODELAGEM DE BIOMASSA FLORESTAL E CÁLCULO DE C...,DURANTE AS ÚLTIMAS DÉCADAS AS FLORESTAS TROPIC...,ACRE,MODELAGEM DE BIOMASSA FLORESTAL E CÁLCULO DE C...,"[0.014655284, 0.17252105, 0.12373036, 0.116068..."
4,07dae75eae08bbf1828e779c70c9d283965fe58880d02f...,ANÁLISE SOCIOECONÔMICA E AMBIENTAL DA CADEIA P...,O BIOMA AMAZÔNICO OCUPA POSIÇÃO DE DESTAQUE NO...,ACRE,ANÁLISE SOCIOECONÔMICA E AMBIENTAL DA CADEIA P...,"[0.1399752, 0.026012428, 0.087050915, 0.055632..."


# 3. Busca dos Top-K Temas



In [ ]:
def encontrar_top_k_temas(producao_embedding, df_temas_com_embeddings, k=5):
    """
    Encontra os k temas mais similares a uma produção acadêmica.

    Args:
        producao_embedding: O vetor de embedding da produção.
        df_temas_com_embeddings: DataFrame com os temas e seus embeddings.
        k: O número de temas a serem retornados.

    Returns:
        Um DataFrame com os top-k temas e suas similaridades.
    """
    # Calcula a similaridade de cossenos entre o embedding da produção e todos os temas
    similaridades = cos_sim(producao_embedding, df_temas_com_embeddings['tema_embedding'].to_list())

    # Adiciona as similaridades ao DataFrame de temas
    df_temas_com_embeddings['similaridade'] = similaridades[0].numpy()

    # Ordena por similaridade e retorna os top k
    top_k = df_temas_com_embeddings.sort_values(by='similaridade', ascending=False).head(k)

    return top_k

# 4. Decisão Final com LLM


In [ ]:
# 1. Instala o Ollama command-line tool
print("Instalando o Ollama...")
# Usamos os.system para uma saída mais limpa e direta no Colab
os.system("curl -fsSL https://ollama.com/install.sh | sh")
print("Instalação do Ollama concluída.")

# 2. Inicia o servidor Ollama em background
# Usamos nohup para garantir que o processo continue rodando de forma independente
os.system("nohup ollama serve &")
print("Iniciando o servidor Ollama... (aguardando 10 segundos para inicialização)")
time.sleep(10)

# 3. Baixa o modelo (se ainda não estiver presente)
print("\nBaixando o modelo llama3.1...")
os.system("ollama pull llama3.1")
print("Modelo pronto para uso.")

# 4. Configura o cliente da OpenAI para usar o endpoint local do Ollama
openai.api_base = "http://localhost:11434/v1"
openai.api_key = "ollama" # Chave fictícia, necessária pela biblioteca
print("\nConfiguração concluída. O ambiente está pronto para usar o LLM.")

Instalando o Ollama...
Instalação do Ollama concluída.
Iniciando o servidor Ollama... (aguardando 10 segundos para inicialização)

Baixando o modelo llama3.1...
Modelo pronto para uso.

Configuração concluída. O ambiente está pronto para usar o LLM.


## 4.2. Construindo o Prompt e Executando o LLM


In [ ]:
def classificar_com_llm(producao, temas_candidatos, modelo_llm="llama3.1"):
    """
    Usa um LLM para classificar a aderência de uma produção a uma lista de temas.
    """
    # Estrutura de resposta desejada em JSON
    estrutura_json = {
        "tema_selecionado": "<nome do tema escolhido da lista>",
        "nivel_aderencia": "<ALTA, MÉDIA ou BAIXA>",
        "justificativa": "<breve explicação para a escolha e o nível de aderência>"
    }

    # Prompt do sistema: define o papel e o formato de saída
    prompt_sistema = f"""
    Você é um especialista em análise de produção científica. Sua tarefa é analisar o texto de uma tese/dissertação
    e determinar qual dos temas estratégicos fornecidos é o mais aderente.
    Sua resposta DEVE ser um objeto JSON válido com a seguinte estrutura:
    {json.dumps(estrutura_json, indent=2)}

    CRITÉRIOS DE ADERÊNCIA:
    - ALTA: O tema é o foco central da pesquisa.
    - MÉDIA: O tema é relevante, mas secundário ou abordado de forma indireta.
    - BAIXA: O tema tem uma relação apenas superficial ou tangencial com a pesquisa.
    """

    # Formatando a lista de temas para o prompt
    temas_formatados = "\n".join([f"- {tema.strip()}" for tema in temas_candidatos])

    # Prompt do usuário: contém os dados específicos da tarefa
    prompt_usuario = f"""
    --- DADOS DA PRODUÇÃO ACADÊMICA ---
    TÍTULO: {producao['nome_producao']}
    RESUMO: {producao['descricao_resumo']}

    --- LISTA DE TEMAS ESTRATÉGICOS CANDIDATOS ---
    {temas_formatados}

    --- TAREFA ---
    Com base nos dados da produção, escolha o tema mais aderente da lista de candidatos,
    defina o nível de aderência e forneça uma justificativa concisa.
    Responda APENAS com o objeto JSON.
    """

    # Executa a chamada para o LLM
    try:
        response = openai.ChatCompletion.create(
            model=modelo_llm,
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": prompt_usuario},
            ],
            temperature=0.0
        )
        resposta_texto = response['choices'][0]['message']['content'].strip()

        # Tenta parsear a resposta como JSON
        return json.loads(resposta_texto)
    except Exception as e:
        print(f"Erro ao processar resposta do LLM: {e}")
        print(f"Resposta recebida: {resposta_texto}")
        return None

#5. Teste do Fluxo

In [ ]:
from codecarbon import EmissionsTracker

#Vamos testar o fluxo com 10 amostras diferentes

# Seleciona 10 amostras aleatórias para teste
amostras_para_teste = amostra_producoes_df.sample(n=10, random_state=42).reset_index(drop=True)

resultados_teste = []

tracker = EmissionsTracker(output_file="emissions_llm_batch.csv")
tracker.start()

print("Iniciando teste com 10 amostras...")

for index, producao_exemplo in amostras_para_teste.iterrows():
    print(f"\n--- Processando Produção {index + 1}/{len(amostras_para_teste)} ---")
    print(f"Título: {producao_exemplo['nome_producao']}")
    print(f"UF do Pesquisador: {producao_exemplo['uf_pesquisador']}")

    # Filtra os temas para corresponder à UF do pesquisador
    temas_da_uf = df_temas[df_temas['uf_tema_info'] == producao_exemplo['uf_pesquisador']].copy()

    # Encontra os temas mais próximos
    embedding_exemplo = producao_exemplo['producao_embedding']
    top_temas_df = encontrar_top_k_temas(embedding_exemplo, temas_da_uf, k=5)

    #print("\nTop 5 Temas Mais Próximos (Recuperados):")
    #display(top_temas_df[['tema', 'similaridade']])

    # Guarda a lista para o próximo passo
    top_k_temas_lista = top_temas_df['tema'].tolist()

    #print("Enviando a tarefa para o LLM...")
    resultado_llm = classificar_com_llm(producao_exemplo, top_k_temas_lista)

    # Encontrar o tema original no dataframe completo 'dados_df'
    tema_original_df = dados_df[dados_df['hash_id'] == producao_exemplo['hash_id']][['tema_id', 'tema']].drop_duplicates()
    tema_original = tema_original_df['tema'].tolist() if not tema_original_df.empty else ["N/A"]


    # Armazenar os resultados
    resultados_teste.append({
        'top 5 temas recuperados': top_temas_df[['tema', 'similaridade']].to_dict('records'),
        'hash_id': producao_exemplo['hash_id'],
        'titulo': producao_exemplo['nome_producao'],
        'uf': producao_exemplo['uf_pesquisador'],
        'tema_original': tema_original,
        'top_5_temas_recuperados': top_temas_df[['tema', 'similaridade']].to_dict('records'),
        'resultado_llm': resultado_llm
    })


emissions: float = tracker.stop()
print(f"\nEmissões de carbono totais para o teste com 10 amostras: {emissions} kgCO₂eq")

print("\n--- Resumo dos Resultados do Teste ---")
for i, res in enumerate(resultados_teste):
    print(f"\nProdução {i+1}:")
    print(f"  Título: {res['titulo']}")
    print(f"  UF: {res['uf']}")
    print(f"  Top 5 Temas Recuperados:")
    for tema_info in res['top 5 temas recuperados']:
        print(f"    - Tema: {tema_info['tema']}, Similaridade: {tema_info['similaridade']}")
    print(f"  Tema(s) Original(is): {res['tema_original']}")
    if res['resultado_llm']:
        print(f"  Tema Selecionado pelo LLM: {res['resultado_llm'].get('tema_selecionado')}")
        print(f"  Nível de Aderência do LLM: {res['resultado_llm'].get('nivel_aderencia')}")
    else:
        print("  Classificação LLM falhou.")

Iniciando teste com 10 amostras...

--- Processando Produção 1/10 ---
Título: A JORNADA DA HEROÍNA APRENDIZ: MOTIVANDO MULHERES EM CURSOS STEM ATRAVÉS DO PODER DE UMA NARRATIVA
UF do Pesquisador: RIO DE JANEIRO

--- Processando Produção 2/10 ---
Título: A GOVERNANÇA E A CONCRETIZAÇÃO DE POLÍTICAS PÚBLICAS NA TUTELA DOS VULNERÁVEIS: MARCO REGULATÓRIO DASLICITAÇÕES NO ÂMBITO DA LEI 14.133/21
UF do Pesquisador: MINAS GERAIS

--- Processando Produção 3/10 ---
Título: CRIANÇAS E ADOLESCENTES COM TRANSTORNO DO ESPECTRO AUTISTA (TEA) E DEFICIÊNCIA INTELECTUAL: O ESTUDO SOBRE A RELAÇÃO DA SITUAÇÃO DE POBREZA E A VIOLÊNCIA SEXUAL INTRAFAMILIAR.
UF do Pesquisador: BAHIA

--- Processando Produção 4/10 ---
Título: O IMPACTO DA MIGRAÇÃO E DAS EXPERIÊNCIAS NOS ABRIGOS DE ACOLHIDA: UM ESTUDO SOBRE TRAUMA PSICOSSOCIAL
UF do Pesquisador: RORAIMA

--- Processando Produção 5/10 ---
Título: CANINDÉ DE SÃO FRANCISCO/SE: ESTRATÉGIAS PARA IMPLANTAÇÃO DO ECOTURISMO
UF do Pesquisador: SERGIPE

--- Processando 